In [30]:
import pandas as pd

df = pd.read_csv("../data/processed/telco_clean_eda.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [31]:
def tenure_group(t):
    if t <= 12:
        return "New"
    elif t <= 36:
        return "Mid"
    else:
        return "Loyal"

df["TenureGroup"] = df["tenure"].apply(tenure_group)

In [32]:
services = [
    "PhoneService","MultipleLines","InternetService",
    "OnlineSecurity","OnlineBackup","DeviceProtection",
    "TechSupport","StreamingTV","StreamingMovies"
]

df["TotalServices"] = df[services].apply(lambda x: (x=="Yes").sum(), axis=1)

In [33]:
df["AvgMonthlyValue"] = df["TotalCharges"] / (df["tenure"] + 1)

In [34]:
contract_map = {
    "Month-to-month": 2,
    "One year": 1,
    "Two year": 0
}

df["ContractRisk"] = df["Contract"].map(contract_map)

In [35]:
df["AutoPay"] = df["PaymentMethod"].apply(
    lambda x: 0 if "automatic" in x.lower() else 1
)

In [36]:
df["EngagementScore"] = (
    df["TotalServices"]*2 +
    df["tenure"]/12 +
    (1-df["ContractRisk"])*2
)

In [37]:
df["HighValue"] = (df["MonthlyCharges"] > df["MonthlyCharges"].median()).astype(int)

In [38]:
df["AtRisk"] = (
    (df["Contract"]=="Month-to-month") &
    (df["tenure"]<12) &
    (df["MonthlyCharges"]>df["MonthlyCharges"].median())
).astype(int)

In [39]:
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,TotalServices,AvgMonthlyValue,ContractRisk,AutoPay,EngagementScore,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_Mid,TenureGroup_New
0,0,1,29.85,29.85,0,1,14.925000,2,1,0.083333,...,False,False,False,False,True,False,True,False,False,True
1,0,34,56.95,1889.50,0,3,53.985714,1,1,8.833333,...,False,False,True,False,False,False,False,True,True,False
2,0,2,53.85,108.15,1,3,36.050000,2,1,4.166667,...,False,False,False,False,True,False,False,True,False,True
3,0,45,42.30,1840.75,0,3,40.016304,1,0,9.750000,...,False,False,True,False,False,False,False,False,False,False
4,0,2,70.70,151.65,1,1,50.550000,2,1,0.166667,...,False,False,False,False,True,False,True,False,False,True


In [41]:
df_encoded.to_csv("../data/processed/telco_features.csv", index=False)